# Load & Compress Data Pipeline

This notebook replicates the exact data loading and transformation pipeline used by the geospatial site scoring application.

**Pipeline stages:**
1. Load all 7 input CSV files (same as `data_transform.py`)
2. Aggregate monthly records → one row per site
3. Join geospatial distance features
4. Engineer features (log transforms, one-hot encoding, relative strength)
5. Export to compressed Parquet for model training

**Input:** 7 CSV files totaling ~1 GB  
**Output:** `site_training_data.parquet` (~8 MB)

In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
from datetime import datetime, date
from typing import Optional, List, Tuple

# Project paths
PROJECT_ROOT = Path('.').resolve()
INPUT_DIR = PROJECT_ROOT / 'data' / 'input'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Input directory:  {INPUT_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'\nInput files:')
for f in sorted(INPUT_DIR.iterdir()):
    if f.is_file() and f.suffix == '.csv':
        size_mb = f.stat().st_size / 1024 / 1024
        print(f'  {f.name:50s} {size_mb:8.1f} MB')

---
## Stage 1: Load All Input Files

The application uses **Polars** (not pandas) for loading because the main CSV is 927 MB / 1.4M rows.
Polars is 10-20x faster than pandas for this workload (ADR-003).

### 1a. Main Site Scores Data

In [ ]:
# Load the main Site Scores CSV — same parameters as data_transform.load_site_scores()
print('Loading Site Scores data...')
site_scores = pl.read_csv(
    INPUT_DIR / 'site_scores_revenue_and_diagnostics.csv',
    infer_schema_length=10000,
    null_values=['', 'NA', 'null', 'Unknown']
)
print(f'Loaded {len(site_scores):,} monthly records for {site_scores["id_gbase"].n_unique():,} unique sites')
print(f'Columns ({len(site_scores.columns)}): {site_scores.columns[:10]} ... (showing first 10)')
print(f'Date range: {site_scores["date"].min()} to {site_scores["date"].max()}')
site_scores.head(3)

### 1b. Auxiliary Geospatial Distance Files

Six auxiliary CSVs provide distance-to-landmark features (in miles).

In [ ]:
# Load auxiliary geospatial data — same as data_transform.load_auxiliary_data()
print('Loading auxiliary geospatial data...\n')

# 1. Nearest site distances
nearest = pl.read_csv(INPUT_DIR / 'nearest_site_distances.csv')
print(f'Nearest sites:         {len(nearest):,} records | cols: {nearest.columns}')

# 2. Interstate distances — some sites have multiple, take minimum distance
interstate_raw = pl.read_csv(INPUT_DIR / 'site_interstate_distances.csv')
interstate = interstate_raw.group_by('GTVID').agg([
    pl.col('distance_to_interstate_mi').min().alias('min_distance_to_interstate_mi'),
    pl.col('nearest_interstate').first().alias('nearest_interstate')
])
print(f'Interstate distances:   {len(interstate):,} unique sites (from {len(interstate_raw):,} raw records)')

# 3. Kroger distances (pre-aggregated — one row per site)
kroger = pl.read_csv(INPUT_DIR / 'site_kroger_distances.csv')
print(f'Kroger distances:      {len(kroger):,} records')

# 4. McDonald's distances (pre-aggregated — one row per site)
mcdonalds = pl.read_csv(INPUT_DIR / 'site_mcdonalds_distances.csv')
print(f"McDonald's distances:  {len(mcdonalds):,} records")

# 5. Walmart distances
walmart = pl.read_csv(INPUT_DIR / 'site_walmart_distances.csv')
print(f'Walmart distances:     {len(walmart):,} records')

# 6. Target distances
target = pl.read_csv(INPUT_DIR / 'site_target_distances.csv')
print(f'Target distances:      {len(target):,} records')

---
## Stage 2: Aggregate Monthly Records → One Row Per Site

The raw data has one row per site per month. We aggregate to one row per site with:
- **Total** revenue and auction metrics across all months
- **Average** metrics per active month
- **Most recent** metadata values (status, location, demographics, etc.)
- **Relative strength** momentum indicators (multi-horizon)

In [ ]:
# Revenue/auction metrics to aggregate
revenue_metrics = [
    'revenue', 'monthly_impressions', 'monthly_nvis',
    'monthly_impressions_per_screen', 'monthly_nvis_per_screen',
    'monthly_revenue_per_screen'
]

# Build aggregation expressions — mirrors data_transform.aggregate_site_metrics()
agg_exprs = [
    # Count of active months
    pl.len().alias('active_months'),
    # Date range
    pl.col('date').min().alias('first_month'),
    pl.col('date').max().alias('last_month'),
    # Most recent site metadata (last value when sorted by date)
    pl.col('gtvid').last().alias('gtvid'),
    pl.col('site_activated_date').last().alias('site_activated_date'),
    pl.col('network').last().alias('network'),
    pl.col('state').last().alias('state'),
    pl.col('county').last().alias('county'),
    pl.col('latitude').last().alias('latitude'),
    pl.col('longitude').last().alias('longitude'),
    pl.col('zip').last().alias('zip'),
    pl.col('dma').last().alias('dma'),
    pl.col('dma_rank').last().alias('dma_rank'),
    pl.col('statuis').last().alias('status'),  # Note: typo in source data ('statuis' not 'status')
    pl.col('program').last().alias('program'),
    pl.col('experience_type').last().alias('experience_type'),
    pl.col('hardware_type').last().alias('hardware_type'),
    pl.col('retailer').last().alias('retailer'),
    pl.col('screen_count').last().alias('screen_count'),
    # Demographics
    pl.col('avg_household_income').last().alias('avg_household_income'),
    pl.col('median_age').last().alias('median_age'),
    pl.col('pct_african_american').last().alias('pct_african_american'),
    pl.col('pct_asian').last().alias('pct_asian'),
    pl.col('pct_female').last().alias('pct_female'),
    pl.col('pct_male').last().alias('pct_male'),
    pl.col('pct_hispanic').last().alias('pct_hispanic'),
    # Brands
    pl.col('brand_fuel').last().alias('brand_fuel'),
    pl.col('brand_restaurant').last().alias('brand_restaurant'),
    pl.col('brand_c_store').last().alias('brand_c_store'),
    # Capability flags
    pl.col('schedule_site').last().alias('schedule_site'),
    pl.col('sellable_site').last().alias('sellable_site'),
    pl.col('c_emv_enabled').last().alias('c_emv_enabled'),
    pl.col('c_nfc_enabled').last().alias('c_nfc_enabled'),
    pl.col('c_open_24_hours').last().alias('c_open_24_hours'),
    pl.col('c_sells_beer').last().alias('c_sells_beer'),
    pl.col('c_sells_diesel_fuel').last().alias('c_sells_diesel_fuel'),
    pl.col('c_sells_lottery').last().alias('c_sells_lottery'),
    pl.col('c_vistar_programmatic_enabled').last().alias('c_vistar_programmatic_enabled'),
    pl.col('c_walk_up_enabled').last().alias('c_walk_up_enabled'),
    pl.col('c_sells_wine').last().alias('c_sells_wine'),
    # Restriction flags
    pl.col('r_lottery').last().alias('r_lottery'),
    pl.col('r_government').last().alias('r_government'),
    pl.col('r_travel_and_tourism').last().alias('r_travel_and_tourism'),
    pl.col('r_retail_car_wash').last().alias('r_retail_car_wash'),
    pl.col('r_cpg_beverage_beer_oof').last().alias('r_cpg_beverage_beer_oof'),
    pl.col('r_cpg_beverage_beer_vide').last().alias('r_cpg_beverage_beer_vide'),
    pl.col('r_cpg_beverage_wine_oof').last().alias('r_cpg_beverage_wine_oof'),
    pl.col('r_cpg_beverage_wine_video').last().alias('r_cpg_beverage_wine_video'),
    pl.col('r_finance_credit_cards').last().alias('r_finance_credit_cards'),
    pl.col('r_cpg_cbd_hemp_ingestibles_non_thc').last().alias('r_cpg_cbd_hemp_ingestibles_non_thc'),
    pl.col('r_cpg_non_food_beverage_cannabis_medical').last().alias('r_cpg_non_food_beverage_cannabis_medical'),
    pl.col('r_cpg_non_food_beverage_cannabis_recreational').last().alias('r_cpg_non_food_beverage_cannabis_recreational'),
    pl.col('r_cpg_non_food_beverage_cbd_hemp_non_thc').last().alias('r_cpg_non_food_beverage_cbd_hemp_non_thc'),
    pl.col('r_alcohol_drink_responsibly_message').last().alias('r_alcohol_drink_responsibly_message'),
    pl.col('r_alternative_transportation').last().alias('r_alternative_transportation'),
    pl.col('r_associations_and_npo_anti_smoking').last().alias('r_associations_and_npo_anti_smoking'),
    pl.col('r_automotive_after_market_oil').last().alias('r_automotive_after_market_oil'),
    pl.col('r_cpg_beverage_spirits_ooh').last().alias('r_cpg_beverage_spirits_ooh'),
    pl.col('r_cpg_beverage_spirits_video').last().alias('r_cpg_beverage_spirits_video'),
    pl.col('r_cpg_non_food_beverage_e_cigarette').last().alias('r_cpg_non_food_beverage_e_cigarette'),
    pl.col('r_entertainment_casinos_and_gambling').last().alias('r_entertainment_casinos_and_gambling'),
    pl.col('r_government_political').last().alias('r_government_political'),
    pl.col('r_automotive_electric').last().alias('r_automotive_electric'),
    pl.col('r_recruitment').last().alias('r_recruitment'),
    pl.col('r_restaurants_cdr').last().alias('r_restaurants_cdr'),
    pl.col('r_restaurants_qsr').last().alias('r_restaurants_qsr'),
    pl.col('r_retail_automotive_service').last().alias('r_retail_automotive_service'),
    pl.col('r_retail_grocery').last().alias('r_retail_grocery'),
    pl.col('r_retail_grocerty_with_fuel').last().alias('r_retail_grocerty_with_fuel'),
]

# Add revenue metric totals
for col in revenue_metrics:
    if col in site_scores.columns:
        agg_exprs.append(pl.col(col).sum().alias(f'total_{col}'))

print(f'Aggregating {len(site_scores):,} monthly records to one row per site...')
print(f'Using {len(agg_exprs)} aggregation expressions')

In [ ]:
# Sort by site and date, then aggregate
df_sorted = site_scores.sort(['id_gbase', 'date'])
site_agg = df_sorted.group_by('id_gbase').agg(agg_exprs)

# Calculate averages per active month
for col in revenue_metrics:
    total_col = f'total_{col}'
    if total_col in site_agg.columns:
        site_agg = site_agg.with_columns(
            (pl.col(total_col) / pl.col('active_months')).alias(f'avg_monthly_{col}')
        )

print(f'Aggregated to {len(site_agg):,} sites')
print(f'Columns: {len(site_agg.columns)}')

# Show status distribution
status_dist = site_agg.group_by('status').agg(pl.len().alias('count')).sort('count', descending=True)
print(f'\nStatus distribution:')
for row in status_dist.iter_rows(named=True):
    pct = row['count'] / len(site_agg) * 100
    print(f'  {row["status"]:15s} {row["count"]:6,} ({pct:.1f}%)')

### 2b. Calculate Relative Strength Indicators (Momentum Features)

RS compares short-term vs long-term performance for revenue and impression metrics.  
Values > 1.0 = upward trend, < 1.0 = declining.  
Three horizons capture short/medium/long-term momentum.

In [ ]:
def calculate_relative_strength(
    df: pl.DataFrame,
    metric_col: str,
    output_col: str,
    short_days: int = 30,
    long_days: int = 90,
    min_observations: int = 3,
) -> pl.DataFrame:
    """Calculate relative strength for a metric (short-term avg / long-term avg)."""
    if metric_col not in df.columns:
        print(f'  Warning: {metric_col} not found, skipping {output_col}')
        return pl.DataFrame({'id_gbase': [], output_col: []})

    if 'date_parsed' not in df.columns:
        df = df.with_columns(pl.col('date').str.to_date().alias('date_parsed'))

    max_date = df['date_parsed'].max()
    short_cutoff = max_date - pl.duration(days=short_days)
    long_cutoff = max_date - pl.duration(days=long_days)

    short_term = (
        df.filter(pl.col('date_parsed') >= short_cutoff)
        .group_by('id_gbase')
        .agg([
            pl.col(metric_col).mean().alias('short_avg'),
            pl.col(metric_col).count().alias('short_count'),
        ])
    )

    long_term = (
        df.filter(pl.col('date_parsed') >= long_cutoff)
        .group_by('id_gbase')
        .agg([
            pl.col(metric_col).mean().alias('long_avg'),
            pl.col(metric_col).count().alias('long_count'),
        ])
    )

    epsilon = 1e-6
    rs_df = short_term.join(long_term, on='id_gbase', how='left').with_columns([
        pl.when(
            (pl.col('short_count') >= min_observations) &
            (pl.col('long_count') >= min_observations)
        )
        .then((pl.col('short_avg') + epsilon) / (pl.col('long_avg') + epsilon))
        .otherwise(None)
        .alias(output_col)
    ]).select(['id_gbase', output_col])

    rs_df = rs_df.with_columns(
        pl.col(output_col).fill_null(1.0).fill_nan(1.0)
    )
    return rs_df


print('Function defined: calculate_relative_strength()')

In [ ]:
# Calculate multi-horizon relative strength features
# Data is MONTHLY, so windows are calibrated for monthly data points:
#   - 95/185 days  → ~3/6 months (short-term)
#   - 185/370 days → ~6/12 months (medium-term)
#   - 370/740 days → ~12/24 months (long-term)
horizons = [(95, 185), (185, 370), (370, 740)]
rs_metrics = [
    ('monthly_impressions', 'Impressions'),
    ('monthly_nvis', 'NVIs'),
    ('revenue', 'Revenue'),
    ('monthly_revenue_per_screen', 'RevenuePerScreen'),
]

# Parse date once
scores_with_date = site_scores.with_columns(pl.col('date').str.to_date().alias('date_parsed'))

all_rs_dfs = []
for short_days, long_days in horizons:
    print(f'\nHorizon: {short_days}/{long_days} days')
    for metric_col, metric_name in rs_metrics:
        output_col = f'rs_{metric_name}_{short_days}_{long_days}'
        rs_df = calculate_relative_strength(
            scores_with_date, metric_col, output_col,
            short_days=short_days, long_days=long_days,
            min_observations=2,
        )
        if len(rs_df) > 0:
            all_rs_dfs.append(rs_df)
            mean_val = rs_df[output_col].mean()
            print(f'  {output_col}: {len(rs_df):,} sites (mean={mean_val:.3f})')

# Join all RS features together
rs_result = all_rs_dfs[0]
for rs_df in all_rs_dfs[1:]:
    rs_result = rs_result.join(rs_df, on='id_gbase', how='full', coalesce=True)

# Fill nulls with 1.0 (neutral)
rs_cols = [col for col in rs_result.columns if col.startswith('rs_')]
for col in rs_cols:
    rs_result = rs_result.with_columns(pl.col(col).fill_null(1.0))

# Join RS features to site aggregates
site_agg = site_agg.join(rs_result, on='id_gbase', how='left')
for col in rs_cols:
    if col in site_agg.columns:
        site_agg = site_agg.with_columns(pl.col(col).fill_null(1.0))

print(f'\nAdded {len(rs_cols)} RS features to site aggregates')
print(f'Site aggregates shape: {site_agg.shape}')

---
## Stage 3: Join Geospatial Distance Features

Join the 6 auxiliary distance datasets to the site-level aggregates via `gtvid` ↔ `GTVID`.

In [ ]:
# Join nearest site distances
site_agg = site_agg.join(
    nearest.select([
        'GTVID',
        'nearest_site',
        pl.col('nearest_site_distance_mi').alias('min_distance_to_nearest_site_mi')
    ]),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join interstate distances
site_agg = site_agg.join(
    interstate,
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join Kroger distances
site_agg = site_agg.join(
    kroger.select(['GTVID', 'min_distance_to_kroger_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join McDonald's distances
site_agg = site_agg.join(
    mcdonalds.select(['GTVID', 'min_distance_to_mcdonalds_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join Walmart distances
site_agg = site_agg.join(
    walmart.select(['GTVID', 'min_distance_to_walmart_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join Target distances
site_agg = site_agg.join(
    target.select(['GTVID', 'min_distance_to_target_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Report join coverage
print('Geospatial feature join coverage:')
for col_name, label in [
    ('nearest_site', 'Nearest site'),
    ('min_distance_to_interstate_mi', 'Interstate'),
    ('min_distance_to_kroger_mi', 'Kroger'),
    ('min_distance_to_mcdonalds_mi', "McDonald's"),
    ('min_distance_to_walmart_mi', 'Walmart'),
    ('min_distance_to_target_mi', 'Target'),
]:
    if col_name in site_agg.columns:
        matched = site_agg.filter(pl.col(col_name).is_not_null()).shape[0]
        pct = matched / len(site_agg) * 100
        print(f'  {label:15s} {matched:6,} / {len(site_agg):,} ({pct:.1f}%)')

print(f'\nPre-cleaned dataset: {site_agg.shape[0]:,} sites × {site_agg.shape[1]} columns')

### Save Pre-cleaned Intermediate (optional checkpoint)

In [ ]:
# Save pre-cleaned dataset as checkpoint
precleaned_path = OUTPUT_DIR / 'site_aggregated_precleaned.parquet'
site_agg.write_parquet(precleaned_path)
size_mb = precleaned_path.stat().st_size / 1024 / 1024
print(f'Saved pre-cleaned checkpoint: {precleaned_path.name} ({size_mb:.1f} MB)')

---
## Stage 4: Prepare Training Dataset

Transform the pre-cleaned data into ML-ready format:
1. Filter to **Active sites only**
2. Remove negative revenue records
3. Drop geographic identifiers (state, county, dma, zip)
4. **Bin high-cardinality categoricals** (retailer → top 30 + Other)
5. **Log-transform** numeric features
6. **One-hot encode** capability and restriction flags

In [ ]:
# Start with the pre-cleaned aggregated data
train_df = site_agg.clone()

# 4a. Filter to Active sites only
original_count = len(train_df)
train_df = train_df.filter(pl.col('status') == 'Active')
print(f'Filtered to Active sites: {len(train_df):,} (from {original_count:,})')

# 4b. Remove negative revenue records
before = len(train_df)
train_df = train_df.filter(pl.col('total_revenue') >= 0)
print(f'Removed {before - len(train_df):,} negative revenue records: {len(train_df):,} remaining')

# 4c. Drop geographic identifiers
geo_cols_to_drop = ['state', 'county', 'dma', 'zip', 'zip_4']
existing_geo_cols = [c for c in geo_cols_to_drop if c in train_df.columns]
train_df = train_df.drop(existing_geo_cols)
print(f'Dropped geographic identifiers: {existing_geo_cols}')

In [ ]:
# 4d. Bin high-cardinality categorical features
def bin_high_cardinality(df: pl.DataFrame, column: str, top_n: int = 30) -> pl.DataFrame:
    """Keep top N values, replace the rest with 'Other'."""
    if column not in df.columns:
        return df
    value_counts = df[column].value_counts().sort('count', descending=True)
    top_values = value_counts[column].head(top_n).to_list()
    original_unique = df[column].n_unique()
    df = df.with_columns(
        pl.when(pl.col(column).is_in(top_values))
        .then(pl.col(column))
        .otherwise(pl.lit('Other'))
        .alias(column)
    )
    new_unique = df[column].n_unique()
    print(f'  Binned "{column}": {original_unique:,} → {new_unique} categories (top {top_n} + Other)')
    return df

train_df = bin_high_cardinality(train_df, column='retailer', top_n=30)
train_df = bin_high_cardinality(train_df, column='brand_c_store', top_n=30)
train_df = bin_high_cardinality(train_df, column='brand_fuel', top_n=10)

In [ ]:
# 4e. Add log transformations
# Sign-preserving: sign(x) * log(1 + |x|) for signed types
numeric_cols_to_log = [
    'total_revenue', 'total_monthly_impressions', 'total_monthly_nvis',
    'total_monthly_impressions_per_screen', 'total_monthly_nvis_per_screen',
    'total_monthly_revenue_per_screen',
    'min_distance_to_nearest_site_mi', 'min_distance_to_interstate_mi',
    'min_distance_to_kroger_mi', 'min_distance_to_mcdonalds_mi',
    'min_distance_to_walmart_mi', 'min_distance_to_target_mi',
    'avg_household_income',
]

log_count = 0
for col in numeric_cols_to_log:
    if col in train_df.columns:
        col_dtype = train_df[col].dtype
        if col_dtype in [pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64]:
            train_df = train_df.with_columns(
                (pl.col(col).cast(pl.Float64) + 1).log().alias(f'log_{col}')
            )
        else:
            train_df = train_df.with_columns(
                pl.when(pl.col(col).cast(pl.Float64) >= 0)
                .then((pl.col(col).cast(pl.Float64) + 1).log())
                .otherwise(-(-pl.col(col).cast(pl.Float64) + 1).log())
                .alias(f'log_{col}')
            )
        log_count += 1

print(f'Added {log_count} log-transformed columns')

In [ ]:
# 4f. One-hot encode capability (c_*) and restriction (r_*) flags
capability_cols = [c for c in train_df.columns if c.startswith('c_')]
restriction_cols = [c for c in train_df.columns if c.startswith('r_')]
other_flags = ['schedule_site', 'sellable_site']
all_flag_cols = capability_cols + restriction_cols + other_flags

encoded_count = 0
for col in all_flag_cols:
    if col not in train_df.columns:
        continue
    col_dtype = train_df[col].dtype
    
    if col_dtype == pl.Boolean:
        train_df = train_df.with_columns(
            pl.col(col).cast(pl.Int8).alias(f'{col}_encoded')
        )
        encoded_count += 1
    elif col_dtype == pl.Utf8:
        unique_vals = train_df[col].unique().to_list()
        if set(unique_vals) <= {'Yes', 'No', 'Unknown', None}:
            train_df = train_df.with_columns(
                pl.when(pl.col(col) == 'Yes').then(1)
                .when(pl.col(col) == 'No').then(0)
                .otherwise(None)
                .cast(pl.Int8)
                .alias(f'{col}_encoded')
            )
            encoded_count += 1
        elif set(unique_vals) <= {'true', 'false', 'True', 'False', None}:
            train_df = train_df.with_columns(
                pl.col(col).str.to_lowercase().eq('true').cast(pl.Int8).alias(f'{col}_encoded')
            )
            encoded_count += 1
        else:
            for val in unique_vals:
                if val is not None:
                    safe_val = str(val).replace(' ', '_').replace('-', '_').lower()
                    train_df = train_df.with_columns(
                        (pl.col(col) == val).cast(pl.Int8).alias(f'{col}_{safe_val}')
                    )
                    encoded_count += 1

# Drop original flag columns (keep encoded versions)
cols_to_drop = [c for c in all_flag_cols if c in train_df.columns]
train_df = train_df.drop(cols_to_drop)

print(f'Created {encoded_count} encoded flag columns, dropped {len(cols_to_drop)} original columns')
print(f'\nFinal training dataset: {train_df.shape[0]:,} sites × {train_df.shape[1]} columns')

---
## Stage 5: Inspect & Export to Parquet

In [ ]:
# Inspect the final dataset
print('=== Final Training Dataset ===')
print(f'Shape: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns')
print(f'\nColumn breakdown:')
log_cols = [c for c in train_df.columns if c.startswith('log_')]
enc_cols = [c for c in train_df.columns if '_encoded' in c]
rs_cols  = [c for c in train_df.columns if c.startswith('rs_')]
total_cols = [c for c in train_df.columns if c.startswith('total_')]
avg_cols = [c for c in train_df.columns if c.startswith('avg_')]

print(f'  Log-transformed:   {len(log_cols)}')
print(f'  One-hot encoded:   {len(enc_cols)}')
print(f'  Relative strength: {len(rs_cols)}')
print(f'  Total metrics:     {len(total_cols)}')
print(f'  Average metrics:   {len(avg_cols)}')

# Null check
null_counts = train_df.null_count()
cols_with_nulls = [(col, null_counts[col][0]) for col in null_counts.columns if null_counts[col][0] > 0]
if cols_with_nulls:
    print(f'\nColumns with nulls ({len(cols_with_nulls)}):')
    for col_name, count in sorted(cols_with_nulls, key=lambda x: -x[1])[:10]:
        print(f'  {col_name}: {count:,} nulls')
else:
    print('\nNo null values found!')

train_df.head(3)

In [ ]:
# Export to Parquet
output_path = OUTPUT_DIR / 'site_training_data.parquet'
train_df.write_parquet(output_path)

# Also save CSV for inspection
csv_path = OUTPUT_DIR / 'site_training_data.csv'
train_df.write_csv(csv_path)

# Report compression stats
parquet_mb = output_path.stat().st_size / 1024 / 1024
csv_mb = csv_path.stat().st_size / 1024 / 1024
input_mb = sum(
    f.stat().st_size for f in INPUT_DIR.iterdir()
    if f.is_file() and f.suffix == '.csv'
) / 1024 / 1024

print(f'=== Compression Results ===')
print(f'Input CSVs total:   {input_mb:8.1f} MB')
print(f'Output CSV:         {csv_mb:8.1f} MB')
print(f'Output Parquet:     {parquet_mb:8.1f} MB')
print(f'Compression ratio:  {input_mb / parquet_mb:.0f}x (vs input CSVs)')
print(f'CSV → Parquet:      {csv_mb / parquet_mb:.1f}x compression')
print(f'\nFiles saved:')
print(f'  {output_path}')
print(f'  {csv_path}')

---
## Verification: Compare with Application Output

Sanity check that our notebook output matches what `data_transform.py` produces.

In [ ]:
# Quick verification: load the parquet we just wrote and inspect
verify_df = pl.read_parquet(output_path)

print(f'Verification read: {verify_df.shape[0]:,} sites × {verify_df.shape[1]} columns')
print(f'\nKey stats:')
print(f'  avg_monthly_revenue: median=${verify_df["avg_monthly_revenue"].median():,.0f}, '
      f'mean=${verify_df["avg_monthly_revenue"].mean():,.0f}')
print(f'  active_months: median={verify_df["active_months"].median():.0f}, '
      f'mean={verify_df["active_months"].mean():.1f}')
print(f'\nAll columns:')
for i, col in enumerate(sorted(verify_df.columns)):
    print(f'  {i+1:3d}. {col} ({verify_df[col].dtype})')